# Day 3: Tests, and the pipeline that runs them

**The coding agent writes a dozen tests in a minute, and every one passes. How many would fail if the filter were wrong?**

The second half of the day asks another: your team's tests pass on every laptop.

From days 1 and 2, used as defined there: the team repository and its protected main, the status filter, a pull request and its review, a generated change, the directing prompt and its done-when, pull request C, and breaking the filter on purpose to see which tests catch it.

The full reading for the day, with its sources: [the course web page](https://talent.catalyte.ai/courses/irs-bpa-to6/week-02/#day-3)

### Today: a test for each half, and a pipeline that runs them.

*Two kinds of test, then one pipeline:*

- Unit tests: the list function, in the service
- A Playwright test: the page, through the browser
- The pipeline: build, test and scan every pull request

<img src="figures/one-filter-crosses-the-page-and-the-service.png" width="864" alt="Two columns. Left, The page, its own pull request: a Status control, one status, such as open; under it, The case list, only that status. Right, The service, its own pull request: The route, passes the status on; under it, The list function, takes an optional status. Orange arrows carry the status from the control to the route, down to the list function, and back to the case list.">

The figure is day 1's: the status filter is one change in two parts, a control on the page and a list function in the service. Today both halves get tests, written with the coding agent: unit tests for the function that picks the cases, and a Playwright end-to-end test for the page.

Sections 3.1 to 3.4 choose, generate and run them, then deal with a test that fails, and one that fails only sometimes. Sections 3.5 to 3.7 add a machine's checks to day 1's reviewed pull request: a build, both test suites and security scans on every pull request, a failing check that blocks the merge, and a passing change moved to the next environment.

*Source: the chapter, parts 4 and 5*

## 3.1 Unit tests that can fail

*The coding agent writes twelve unit tests for the filter, and all pass first time. What does that tell you?*

### A unit test is fast and reliable, and blind to the page's fault.

<img src="figures/each-fault-and-the-test-that-reaches-it.png" width="864" alt="Three spans over three parts. Top, in orange, A test through the browser: the only one that reaches the page, spanning all three parts; then A test of the route, spanning the route and the list function; then A unit test of the function, spanning the list function only. Below, left to right, with arrows carrying the status: The page, fault: sends the status under a name the route does not read: every case comes back; The route, fault: drops the status before the function sees it; The list function, fault: open returns a closed case; an unknown status raises an error.">

The claim in full: a unit test of the list function catches the function's own faults; only a test through the browser reaches the page, where the status can be sent under a name the route does not read.

A unit test checks one piece of code on its own, which is its strength and its limit. Google's book on its engineering practice: "A unit test is like a problem in theoretical physics: ensconced in a vacuum, neatly hidden from the mess of the real world, which is great for speed and reliability but misses certain defect categories."

Filtering by open returns a closed case, or an unknown status raises an error instead of returning no cases: a unit test of the function catches both.

Fowler, of end-to-end tests: "They are notoriously flaky and often fail for unexpected and unforeseeable reasons. Quite often their failure is a false positive." His rule of thumb for each new test: "Push your tests as far down the test pyramid as you can". For the case list: several unit tests of the function, and one Playwright test that chooses a status and sees only those rows.

### A test that passes first time agrees with the code as it is.

*TestGen-LLM keeps only passing tests.*

- It cannot tell a found bug from a wrong assertion
- One kept test had no assertion; review caught it
- One agent, filter and tests: they can share an error

<img src="figures/keep-the-tests-that-can-fail.png" width="864" alt="Top, 12 generated tests, all passing, drawn as a stack, with two arrows down. Left, grey and dashed: Kept: it passes. TestGen-LLM&#x27;s rule: it agrees with the code as it is, faults included. Right, orange: Kept: it fails on a broken filter. You wrote the expected values, then broke the filter on purpose.">

When one coding agent writes both the filter and its tests, a review of research on tests in software engineering with large language models warns: "Agreement can reflect a shared interpretation error." So you write the expected values, from the done-when (section 2.3 of the chapter), and the agent may write the rest.

*Source: Alshahwan and others, Automated Unit Test Improvement using Large Language Models at Meta; the chapter, section 4.2*

### Build the cases in a fixture, so no test changes another's.

Build the cases in a pytest fixture, a setup function that a test requests by naming it as an argument, rebuilt for every test by default, so no test can change another's cases.

The expected list, ['open', 'open'], is the value you wrote, not one the agent read off the code.

**Check:** Twelve generated tests pass first time. What does that tell you?

- **A.** The filter is correct
- **B.** They agree with the code as it is, faults included
- **C.** Each would fail if the filter were wrong
- **D.** There are enough of them

<details><summary>Answer</summary>

**B.** A test kept because it passes asserts what the code does today; only a break on purpose shows it can fail.

</details>

**Discuss:** What do twelve passing tests tell you?

- Which mistakes in the filter can a test of the function catch, and which only a test through the browser?
- The coding agent writes twelve unit tests and all pass on their first run. What does that tell you about them?
- Why do you write the expected values yourself?

Exercise slot: Write unit tests that fail when the filter is wrong

## 3.2 Find elements as a user would

*The agent's test finds the third row under #cases and sleeps. What do you change?*

### Change the locator and the sleep before you accept it.

### Find rows as a user would, and let the assertion wait.

from playwright.sync_api import Page, expect

As for the sleep: before an action Playwright waits until the element is visible, stable, able to receive events and enabled, and its assertions wait as well: "Playwright assertions are designed in a way that they describe the expectations that need to be eventually met."

<img src="figures/click-waits-until-every-check-passes.png" width="864" alt="Left, locator.click(), the action your test asks for, with an arrow into a frame: Playwright auto-waits until the element: resolves to exactly one element; is Visible; is Stable: not animating; Receives Events: not obscured; is Enabled. Under the frame a loop arrow: not yet: it waits and checks again. Two arrows leave the frame: all pass, to The click runs, every check passed; timeout, to a dashed box, TimeoutError, the checks did not pass within the timeout.">

*Before a click, Playwright waits until the element passes every check, then clicks; if the checks do not pass within the timeout, the click fails with a TimeoutError.*

The claim in full: before a click, Playwright waits until the element passes every check, then clicks; if the checks do not pass within the timeout, the click fails with a TimeoutError.

A timeout is an action whose checks never passed: "If the required checks do not pass within the given timeout, action fails with the TimeoutError." That error is where section 3.4 starts.

The figure was drawn from Playwright's page on auto-waiting, which also lists the first check, that the locator resolves to exactly one element.

**Check:** Which locator would you accept for the case rows?

- **A.** The third table row under #cases
- **B.** get_by_test_id('case-row'), a test id the page keeps
- **C.** The row's position, read after a two-second sleep
- **D.** Whichever locator the coding agent chose

<details><summary>Answer</summary>

**B.** Find elements as a user would, or by a test id kept for tests: a locator tied to page structure fails when it changes.

</details>

**Discuss:** What do you change before you accept it?

- The coding agent's end-to-end test finds the third table row under #cases, sleeps, and reads the row's text. What do you change before you accept it?
- Why would a test tied to the page's structure fail when nothing a user sees has changed?

Exercise slot: Make a generated Playwright test find elements as a user would

## 3.3 Run headless, read the trace

*The Playwright test opens no window, on your machine or in the pipeline. When it fails, how do you see what it did?*

### Headless: no window, and it is the default.

*"--headed: Run tests in headed mode (default: headless)."*

- The same on your machine and in the pipeline
- Run the fast unit suite first
- Then the Playwright suite, keeping a trace of each test that fails

Headless means the browser runs without a window, and it is the default: "--headed: Run tests in headed mode (default: headless)." That is the same on a laptop and in the pipeline, so a test that fails headless leaves nothing to watch: you see what it did only if the run kept a record.

### A failed test leaves a trace.zip; show-trace replays it.

<img src="figures/a-failed-test-leaves-its-trace.png" width="864" alt="Left to right. Unit suite first, pytest tests/unit: the fast one. Then: Playwright suite, headless, pytest tests/e2e --tracing=retain-on-failure. Two arrows leave it. Pass, grey, to a dashed box: A test passes, its trace is removed. Fail, orange, to a file, trace.zip, under test-results: the failed test&#x27;s trace, and from it, in blue, playwright show-trace: replays the failed run, action by action.">

**Check:** A test failed in a headless run. Where do you see what it did?

- **A.** Rerun it with a window until it fails again
- **B.** Its trace.zip under test-results, in playwright show-trace
- **C.** In the failure message alone
- **D.** Nowhere: a headless run keeps no record

<details><summary>Answer</summary>

**B.** --tracing=retain-on-failure keeps the trace of each failed test; show-trace replays it action by action.

</details>

**Discuss:** How do you see what a windowless test did?

- The Playwright test opens no window, on your machine or in the pipeline. When it fails, how do you see what it did?
- Why keep the traces of failed tests only?

Exercise slot: Run both suites headless and replay a failed test's trace

## 3.4 A failing test, a flaky test

*After a teammate's change, the filter test fails. Is the page broken, or the test?*

The section's answer: the two need opposite repairs, so find out before changing anything.

### The trace decides: fix the test, or the test did its job.

<img src="figures/the-trace-says-which-to-repair.png" width="864" alt="Left, The filter test fails, after a teammate&#x27;s change, with an arrow to Read the trace, in blue: its log: which check timed out; its network requests: what the page sent and what came back. Two arrows leave it. Upper: A timeout, the Status control is there under a new name, Case status, and the rename was meant, then The test is at fault: update the locator. Lower: A failed count, the request carried status=open, and the service returned every case, then, in orange, The application is at fault: the test did its job.">

A timeout is an action whose checks never passed: "If the required checks do not pass within the given timeout, action fails with the TimeoutError." The trace says which check: its log shows Playwright "waiting for element to be visible, enabled and stable", and its network requests show what the page sent and what came back.

Upper row: the Status control is there under a new name, Case status, and the rename was meant: the test is at fault; update the locator.

### Be slow to accept an automatic repair: it can bury a regression.

**What a healer does**

- Suggests a patch: locator update, wait adjustment, data fix
- Re-runs the test until it passes or guardrails stop it
- Ends in "A passing test, or a skipped test"

**What it can bury**

- The failed count: status=open sent, every case back
- A pass or a skip both hide that regression

**What you refuse**

- A changed expected count
- A longer wait
- A skip, until the trace says why
- The repair's commit message states the cause

Playwright's healer agent, documented for its Node.js test runner, on a failure "Suggests a patch (e.g., locator update, wait adjustment, data fix)" and "Re-runs the test until it passes or until guardrails stop the loop". Its result, in Playwright's words: "A passing test, or a skipped test if the healer believes that functionality is broken."

### A flaky test costs the meaning of red: find its cause.

*Do not retry it until it passes.*

- Red: a bug, or the flakiness?
- Reruns "might therefore mask actual bugs"
- Look first where the studies found the causes

<img src="figures/where-flakiness-comes-from.png" width="864" alt="Two studies, each with bars on a scale of 0 to 100% of its flaky tests, the full scale a dashed track. Top: Python projects, 7,571 flaky tests: order dependencies 59%, in blue; infrastructure 28%; networking, randomness 13%. Bottom: User-interface tests, 235 flaky, web and Android: async waits 45%, in blue.">

Fowler: "A test is non-deterministic when it passes sometimes and fails sometimes, without any noticeable change in the code, tests, or environment." Such a flaky test costs you the meaning of red: "when they go red, you have no idea whether its due to a bug, or just part of the non-deterministic behavior." Reruns hide that, and a study of Python projects names the risk, as an argument rather than a measurement: "Rerunning flaky tests might therefore mask actual bugs."

Gruber and others ran the tests of 22,352 Python projects 400 times each and found 7,571 flaky tests: "59 % of all flakiness we observed was caused by order dependencies, 28 % by infrastructure flakiness, and 13 % mostly and to equal degrees by the use of networking and randomness APIs." Among 235 user-interface tests, web and Android, already reported or fixed as flaky: "We have found the root cause for a significant portion (45%) of the flaky tests analyzed arise from issues with async wait mechanisms." The bars are each study's share of its own flaky tests.

*Source: Fowler, Eradicating Non-Determinism in Tests; Gruber and others; Romano and others; the chapter, section 4.6*

### Two flaky symptoms on the case list: cause and fix.

| Symptom | Cause | Fix |
| --- | --- | --- |
| The page test counts rows before the filtered list arrives | Asynchronous wait | An assertion that waits; never a bare sleep |
| A unit test passes alone and fails after another | Order dependency, such as day 1's shared default list | Keep tests isolated from each other |

The page test counts rows before the filtered list arrives: an asynchronous wait; the fix is an assertion that waits (section 3.2), and Fowler's rule: "Never use bare sleeps to wait for asynchonous responses: use a callback or polling." A unit test passes alone and fails after another: an order dependency, such as the shared default list from day 1 (section 2.1 of the chapter); Fowler: "Keep your tests isolated from each other, so that execution of one test will not affect any others."

**Check:** status=open sent, every case back. What do you repair?

- **A.** The locator: update it
- **B.** The expected count: change it to match
- **C.** The application: the test did its job
- **D.** Nothing: retry until it passes

<details><summary>Answer</summary>

**C.** The request carried the status and the service ignored it: the fault is the application's. A new count would bury it.

</details>

**Discuss:** Is the page broken, or the test?

- After a teammate's change the filter test fails. Is the page broken, or the test, and how do you find out?
- The filter test fails about one run in ten and passes when run again. Do you let the pipeline retry it until it passes?
- What would make you refuse a repair that a tool proposed?

Exercise slot: Test or application at fault? Then fix a flaky test at its cause

## 3.5 Build each change, then promote it

*The service half passes on its author's laptop. What could another machine find?*

### A pipeline starts clean, so it finds what your laptop supplied.

*Each job runs on a fresh runner.*

- CI's goal: catch problematic changes as early as possible
- Branches that never join main: only semi-integration
- Keep the build to about ten minutes

<img src="figures/the-laptop-supplies-what-the-runner-lacks.png" width="864" alt="A table of ticks with two columns, Laptop and Runner, fresh, the runner in blue. In an orange band headed Supplied unasked, three rows the laptop has and the fresh runner does not: A package installed months ago; A file never committed; A different Python. Below the band, one row both have: What is committed, and the Python you name.">

Its purpose, in Google's words: "Simply put, the fundamental goal of CI is to automatically catch problematic changes as early as possible." CI, continuous integration, is more than a check on a pull request. Fowler calls building a branch that has not joined the mainline semi-integration, which "doesn't prevent branches from diverging, conflicts from festering, and all the problems of low-frequency integration." That is why day 1's branches live for hours.

*Source: GitHub Docs, Workflow syntax for GitHub Actions; Software Engineering at Google, chapter 23; Fowler, Continuous Integration*

### Build each pull request to main, and main after each merge.

| In the workflow | What it does |
| --- | --- |
| on: pull_request, branches: main | Runs on each pull request to main |
| on: push, branches: main | Runs on main after each merge |
| runs-on: ubuntu-latest | Each job: a fresh instance of the runner image |
| python-version: '3.12' | Names the Python: "helps make your dependencies explicit" |
| run: pip install -r requirements.txt | Installs the packages requirements.txt lists |

Run it on each pull request to main, and on main after each merge, as Fowler's service does: "Every time the mainline receives a commit, the CI service checks out the head of the mainline into an integration environment and performs a full build." In GitHub Actions, GitHub's pipeline service, the chapter's workflow:

name: case-list checks on: pull_request: branches: - main push: branches: - main jobs: checks: runs-on: ubuntu-latest steps: - uses: actions/checkout@v6 - uses: actions/setup-python@v5 with: python-version: '3.12' - run: pip install -r requirements.txt

### Promote once main's checks pass and a teammate approves.

<img src="figures/promote-after-the-checks-pass.png" width="864" alt="Along the top, an arrow: Each stage provides increasing confidence, usually at the cost of extra time. Four stages, left to right. Pull request to main: the checks job runs on a fresh runner. Merge, to Merged into main: the checks run again, on main&#x27;s head. Checks pass, in orange, to The deploy job: needs: checks waits for this run&#x27;s checks. Approved, to Staging, in blue: admits only main; a teammate approves; one deployment at a time. Below, a dashed path from the pull request to the deploy job, crossed out: a pull request: if: push keeps it off.">

The claim in full: a passing change is promoted stage by stage: checked on its pull request, checked again on main, then deployed to staging only after that run's checks pass, and only when a teammate approves.

Fowler's answer to fast builds and slow, thorough tests: "A deployment pipeline is a way to deal with this by breaking up your build into stages. Each stage provides increasing confidence, usually at the cost of extra time." To promote a change is to move it to the next stage once the stage before has passed.

GitHub models a stage as an environment, "a general deployment target like production, staging, or development." An environment carries rules: "You can use deployment protection rules to require a manual approval, delay a job, or restrict the environment to certain branches." Which rules a repository can use depends on its GitHub plan.

The second job in the workflow: deploy: needs: checks if: github.event_name == 'push' runs-on: ubuntu-latest environment: staging concurrency: staging steps: - uses: actions/checkout@v6 - run: echo the reference application's deployment step goes here

needs: checks holds the deployment until the same run's checks pass, so what reaches staging is the commit whose checks just passed on main; the if line keeps it off pull requests; and concurrency allows one deployment in progress at a time.

**Check:** Why can a pipeline build find what the laptop never will?

- **A.** It runs the tests faster
- **B.** Each job runs in a fresh instance of a runner image
- **C.** It always uses the newest Python
- **D.** It skips the slow tests

<details><summary>Answer</summary>

**B.** A fresh runner lacks what the laptop supplied unasked: a package installed months ago, a file never committed, a different Python.

</details>

**Discuss:** What could another machine find that your laptop never will?

- The service half passes every test on its author's laptop. What could a build on another machine find?
- The filter is on main and main's checks are green. What moves it to where it is tried next, and who may let it through?

Exercise slot: Build every pull request, and promote a passing change

## 3.6 A failing check blocks the merge

*The Playwright test fails in the pipeline. What stops someone pressing merge?*

### Your laptop's commands, plus the browsers and the traces.

| Step in the workflow | Why |
| --- | --- |
| run: pytest tests/unit | Unit tests first: "only fast, reliable ones" run before a change goes in |
| run: python -m playwright install --with-deps | The browsers your laptop already had |
| run: pytest tests/e2e --tracing=retain-on-failure | The Playwright tests, keeping the traces of failures |
| uses: actions/upload-artifact@v4 | Keeps test-results/ as an artifact, in a trusted store only |

"You can use the same commands that you use locally to build and test your code." Two things are missing: the browsers and a running service. Playwright's own example installs the browsers, runs the tests with the tracing from section 3.3 and keeps the traces as an artifact of the run.

The example does not start the application under test, so the end-to-end tests need a fixture, or the job a step, that does. Google's rule for what runs before a change is submitted is "only fast, reliable ones": the unit tests go first, and the handful of Playwright tests from section 3.1 is what lets them run on every pull request.

Handle it as data: "If you upload reports and traces as part of your CI workflow, make sure that you only upload them to trusted artifact stores, or that you encrypt the files before upload."

The steps in full: - run: pytest tests/unit - run: python -m playwright install --with-deps - run: pytest tests/e2e --tracing=retain-on-failure - uses: actions/upload-artifact@v4 if: ${{ !cancelled() }} with: name: playwright-traces path: test-results/

### A red check blocks nothing; a required check on main does.

<img src="figures/what-a-required-check-lets-through.png" width="864" alt="Three columns: The job reports; Required check, protected main; The pull request. Three rows. Success: passes; the pull request can merge, in blue. Failure: blocks; the pull request is refused. In orange, Skipped by its condition, it reports &quot;Success&quot;: passes; the pull request can merge, unchecked, in blue.">

### The only proof that a check blocks is a change it blocked.

*Seed a fault only a browser finds.*

- The status sent under a name the route does not read
- Link the refused run from the pull request
- Keep strict: changes can pass apart, fail together

<img src="figures/a-seeded-regression-proves-the-block.png" width="864" alt="Two boxes side by side over a third. Left, in blue: Unit tests: all pass, the fault is in the page, which they never reach; a dashed line down, blocks nothing. Right, in orange: The Playwright test fails, it goes through the page; an orange arrow down to: The merge is refused, that run, linked from the pull request, is the evidence.">

Every unit test passes, the Playwright test fails, the merge is refused, and that run, linked from the pull request, is the week's evidence.

The default, strict, also makes a branch catch up with main before it merges; the looser setting saves builds, but "Status checks may fail after you merge your branch if there are incompatible changes with the base branch." That is day 1's sorted and filtered list: two changes that pass apart and fail together.

*Source: the chapter, section 5.4; GitHub Docs, About protected branches*

**Check:** A required job is skipped by its condition. Can it merge?

- **A.** No: a required job must run
- **B.** No: it waits until the job runs
- **C.** Yes: a skipped job reports Success
- **D.** Only if an administrator approves

<details><summary>Answer</summary>

**C.** Required checks pass on successful, skipped or neutral, and a skipped job reports Success: it blocks nothing.

</details>

**Discuss:** The red check, or something else?

- A pull request breaks the filter on the page and the Playwright test fails in the pipeline. What stops someone pressing merge?
- How can a required job that is skipped let a pull request through?
- How would you prove that your check blocks a merge?

Exercise slot: Make a failing check block the merge

## 3.7 Scan, and decide what it means

*Pull request C reaches the pipeline. What could a scanner find in it?*

### Three scans, each reading a different part of the change.

| Scan | What it reads |
| --- | --- |
| Code analysis | The Python in the change: Bandit, or GitHub's code scanning with CodeQL or another tool |
| Secret scanning | The entire Git history on all branches, for hardcoded credentials |
| Dependency checks | The packages installed, against databases of known vulnerabilities |

Code analysis reads the Python in the change: "Bandit is a tool designed to find common security issues in Python code." GitHub's code scanning does the same across a repository, with its CodeQL engine or another tool. Secret scanning: "Secret scanning scans your entire Git history on all branches of your repository for hardcoded credentials, including API keys, passwords, tokens, and other known secret types." Dependency checks read the packages installed, which "must be continually monitored against databases of known vulnerabilities" (NIST).

### A finding is a claim to check: fix it, record it, or silence it.

<img src="figures/each-finding-is-a-claim-to-check.png" width="864" alt="Left, bandit -r, over the service&#x27;s code, on pull request C; under it, dashed: Silent on the query, told you about the scan, not the query. An arrow to A finding, graded by severity and confidence, then to Review it, in orange: a claim to check: some are false positives, some insignificant here. Three arrows leave the review: Fix it, a real problem, in blue; Record and triage it, where the team tracks its work, in gold; Silence it: # nosec, naming the test and why.">

The claim in full: a scan's finding is a claim to check: reviewed, it is fixed, recorded and triaged, or silenced with a # nosec that names the test and why; and a scan that is silent has told you about the scan.

Add a step that runs bandit -r over the service's code, run it on pull request C, and see what it says about the string-built query.

A finding is a claim to check, as the agent's description was on day 2: "In almost all tools, some warnings are false positives, and some are insignificant in the context of the software." Bandit grades each finding by severity and confidence, and the team picks the level it acts on. A finding is silenced in the code only when "the line has been reviewed and the issue is a false positive or acceptable for some other reason": on this team, a # nosec comment that names the test it silences and why, read in review like any other line. Every other finding is recorded and triaged where the team tracks its work, as NIST's secure development framework asks.

**Check:** bandit -r says nothing about C's string-built query. So?

- **A.** The query is safe
- **B.** You know about the scan, not about the query
- **C.** C is ready to merge
- **D.** The query needs a # nosec comment

<details><summary>Answer</summary>

**B.** A scan that is silent there has told you about the scan, not the query: the query still needs reading.

</details>

**Discuss:** Who decides what a finding means?

- Pull request C, with its query built from a string and its new package, reaches the pipeline. Which of its problems could a scanner find?
- Who decides what a finding means?
- When is a # nosec comment acceptable?

Exercise slot: Scan a change and decide what each finding means

## Close

*Key points, and what day 4 builds*

### Keep the tests that can fail, and make a failing one block.

- Keep a generated test only if it fails on a broken filter
- Find elements as a user would; let assertions wait
- Read a failure's trace before you repair; fix a flaky test's cause
- Build on a fresh runner; promote once the checks pass
- A required check blocks; a finding is a claim to check

### Keep the tests that can fail.

- Day 4: a return as XML
- Its parts, and the paths to its values
- A Python reader whose tests come first

Day 4: one sample return's parts and the paths to its values, then a Python reader whose tests and bad-input fixtures come first; the week's labs close the day.